# AI Exposure or Post-Pandemic Employment Adjustment? Explaining the Decline in Early-Career Hiring

## Author: Viet Nguyen

This is the main thesis notebook. The two Python files prepare the large source data; this notebook contains the research decisions, merge, diagnostics, figures, and models. Build and check one section at a time.

## Introduction

Generative artificial intelligence has created growing concern about how the labor market may change, especially for workers beginning their careers. Recent research has found weaker early-career hiring in industries with greater exposure to generative AI. However, a decline in hiring after 2022 does not necessarily mean that AI caused the decline. Some highly AI-exposed industries also experienced unusually strong employment growth during the COVID-19 pandemic, and their later reduction in hiring may partly reflect an adjustment from that earlier growth. Because these explanations can occur at the same time, it is important to consider post-pandemic employment adjustment when examining the relationship between AI exposure and early-career hiring.

This project asks: **After accounting for employment growth above pre-COVID trends, is greater generative AI exposure associated with a larger decline in hiring among workers ages 22 to 24?** To study this question, I combine quarterly hiring and employment data from the U.S. Census Bureau’s Quarterly Workforce Indicators with occupation-level AI exposure scores developed by Eloundou et al. I use pre-pandemic ACS PUMS employment data to translate the occupation-level scores into national industry-level exposure measures.

The study focuses on three main measures:

- **Early-career hiring**: the quarterly hiring rate for workers ages 22 to 24 within each state-industry group.
- **Generative AI exposure**: an industry-level measure of how much the occupations within an industry could potentially be affected by generative AI.
- **Employment above the pre-COVID predicted trend**: the percentage difference between actual employment in 2022 Q1 through Q3 and the employment predicted from each state-industry group’s 2015–2019 trend.

For the employment-trend measure, I estimate a separate pre-COVID trend for each state-industry group using all-age employment. The trend includes a linear time component and quarter indicators for regular seasonal differences. I then compare predicted employment with actual employment during the first three quarters of 2022. The main hiring analysis defines the post period as beginning in 2023 Q1 and excludes 2022 Q4 as a transition quarter.

This is an observational research project, so the results will be interpreted as associations rather than proof that generative AI caused changes in hiring. The AI-exposure measure represents the potential for workers’ tasks to be affected by generative AI; it does not measure whether firms actually adopted AI or replaced workers. Similarly, employment above its predicted trend does not prove that an industry overhired during the pandemic. The goal is to determine whether AI exposure remains associated with changes in early-career hiring after accounting for one measurable form of post-pandemic employment adjustment.

## Data Preparation

For the data preparation part, I use two Python files: **qwi_pipeline.py** and **prepare_ai_exposure.py**. These programs handle the API requests and large-data processing that would be difficult to manage directly in a notebook. The first program downloads the Quarterly Workforce Indicators data from the U.S. Census Bureau. The second program uses ACS PUMS employment records and occupation-level AI exposure scores to create an industry-level AI exposure measure. The smaller prepared files are then loaded into the notebook, where I will check the data, construct the employment-trend measure, combine the research variables, and perform the analysis.

The **qwi_pipeline.py** program downloads quarterly hiring and employment data beginning in 2015 Q1. It collects observations for workers ages 22 to 24 and for workers of all ages. The younger age group is used to measure early-career hiring, while the all-ages group is used to estimate employment trends. The extraction is limited to private-sector employment, all sexes, unadjusted quarterly values, and the state-level jurisdictions with sufficient data. Alaska is excluded because its QWI series ends in 2016 Q2, and Michigan is excluded because its series ends in 2021 Q4. The program preserves the original Census responses, variable definitions, industry labels, and status flags before creating a cleaned three-digit industry file. Missing and Census-suppressed values remain missing rather than being replaced with zero.

The **prepare_ai_exposure.py** program creates the industry AI exposure measure. It reads the national 2015–2019 ACS PUMS person data and keeps employed private wage-and-salary workers. Workers are included when their employment-status code is 1 or 2 and their class-of-worker code is 1 or 2. Each person is weighted using the ACS person weight, `PWGTP`, so that the resulting occupation shares represent the national employed population rather than the number of survey records. The program then connects ACS occupation codes to the occupation-level GPT-4 beta exposure scores developed by Eloundou et al. Finally, it calculates a person-weighted average AI exposure score for each industry.

At first, I planned to construct the research data at the four-digit industry level. However, the ACS, occupation-exposure, and QWI files use different industry and occupation classifications. The ACS person data use 2017-based industry codes and 2018-based occupation codes, while the current QWI data use a newer industry classification. Some ACS codes connect to multiple possible industries or occupations with different AI exposure scores. Using these ambiguous matches would require assumptions that could create artificial precision in the final exposure measure.

To address this problem, I changed the working industry level from four digits to three digits and downloaded the three-digit QWI series directly from Census. I did not create three-digit hiring rates by averaging four-digit rates. The matching program only assigns an industry or occupation when the official crosswalk provides a clear link. An ACS occupation receives an AI exposure score only when its detailed occupation members lead to the same available exposure score. Ambiguous and unmatched records remain unassigned instead of receiving an invented score.

This conservative approach produces AI exposure estimates for 80 of the 96 three-digit industries in the QWI data. Approximately 42.2% of eligible ACS worker weight can be linked jointly to both a three-digit industry and a usable occupation exposure score. This coverage is an important limitation and will be reported in the analysis. The resulting industry scores measure potential exposure to generative AI based on the occupations employed within each industry; they do not measure whether firms actually adopted AI or whether workers were replaced.

The two programs produce the prepared QWI and industry-exposure files needed for the notebook. They do not estimate employment trends, create the final analysis panel, generate figures, or run statistical models. Those steps remain in the notebook so that each research decision, exclusion, diagnostic check, and result can be reviewed and explained clearly.


### Setup and input check

In [3]:
from pathlib import Path
import pandas as pd

# Find the project root
project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / ".git").exists():
    project_root = project_root.parent

if not (project_root / ".git").exists():
    raise FileNotFoundError(
        "Open this notebook from inside the senior-research project."
    )

# Set data paths
qwi_path = project_root / "data/interim/qwi/qwi_research_ind3.csv"
exposure_path = (
    project_root
    / "data/interim/ai_exposure/industry_ai_exposure_3digit.csv"
)

# Check and preview the data
for path in (qwi_path, exposure_path):
    status = "FOUND" if path.exists() else "MISSING"
    print(status, path.relative_to(project_root))

qwi_preview = pd.read_csv(qwi_path, nrows=5)
exposure_preview = pd.read_csv(exposure_path, nrows=5)

display(qwi_preview)
display(exposure_preview)

FOUND data/interim/qwi/qwi_research_ind3.csv
FOUND data/interim/ai_exposure/industry_ai_exposure_3digit.csv


,Emp,EmpEnd,EmpTotal,HirA,HirAEnd,Sep,SepBeg,HirAEndR,SepBegR,sEmp,...,seasonadj,ind_level,state,year,quarter,period,state_name,age_group_name,ownership_name,industry_name
0,125,156,181,56,47,25,15,0.335,0.107,1,...,U,3,1,2015,1,2015Q1,Alabama,22-24,All Private,Crop Production
1,147,164,238,91,52,74,34,0.334,0.219,1,...,U,3,1,2015,2,2015Q2,Alabama,22-24,All Private,Crop Production
2,163,155,208,44,31,53,40,0.195,0.252,1,...,U,3,1,2015,3,2015Q3,Alabama,22-24,All Private,Crop Production
3,157,130,203,46,18,72,44,0.125,0.307,1,...,U,3,1,2015,4,2015Q4,Alabama,22-24,All Private,Crop Production
4,130,161,195,64,47,34,17,0.323,0.117,1,...,U,3,1,2016,1,2016Q1,Alabama,22-24,All Private,Crop Production


,industry,industry_name,ai_exposure,ai_exposure_coverage,eligible_person_weight,matched_person_weight
0,111,Crop Production,0.331394,0.275380,745007.0,205160.0
1,112,Animal Production and Aquaculture,0.338967,0.293680,320236.0,94047.0
2,113,Forestry and Logging,0.290433,0.215166,68338.0,14704.0
3,114,"Fishing, Hunting and Trapping",0.147798,0.744731,21448.0,15973.0
4,115,Support Activities for Agriculture and Forestry,0.273636,0.407574,113675.0,46331.0


### Load data

In [4]:
# Preserve Census identifiers as text
qwi = pd.read_csv(
    qwi_path,
    dtype={
        "state": "string",
        "industry": "string",
        "agegrp": "string",
        "ownercode": "string",
        "sex": "string",
        "seasonadj": "string",
        "ind_level": "string",
        "time": "string",
        "period": "string",
    },
)

exposure = pd.read_csv(
    exposure_path,
    dtype={"industry": "string"},
)

print("QWI shape:", qwi.shape)
print("AI exposure shape:", exposure.shape)

QWI shape: (375068, 33)
AI exposure shape: (96, 6)


### QWI Data Check

The QWI dataset contains the hiring and employment measures used in this study. Before constructing the research variables, I check that the file contains the intended age groups, private-sector observations, three-digit industries, and unadjusted quarterly data. I also check the time coverage and confirm that each state-industry-age-quarter observation is unique. Missing values are counted but not removed because some QWI values are suppressed by Census.


In [7]:
# Confirm the QWI research selections
research_settings = {
    "ind_level": {"3"},
    "ownercode": {"A05"},
    "sex": {"0"},
    "seasonadj": {"U"},
    "agegrp": {"A00", "A03"},
}

for column, expected in research_settings.items():
    actual = set(qwi[column].dropna().unique())
    print(f"{column}: {sorted(actual)}")
    assert actual == expected, f"Unexpected values in {column}: {actual}"


# Check coverage
print("\nRows:", len(qwi))
print("States:", qwi["state"].nunique())
print("Industries:", qwi["industry"].nunique())
print("Period:", qwi["period"].min(), "to", qwi["period"].max())


# Check that each observation is unique
qwi_key = ["state", "industry", "agegrp", "period"]
duplicate_rows = qwi.duplicated(qwi_key, keep=False).sum()

print("Duplicate key rows:", duplicate_rows)
assert duplicate_rows == 0, "The QWI file contains duplicate observations."


# Check missing values in the main research measures
missing_summary = pd.DataFrame(
    {
        "variable": ["EmpEnd for all ages", "HirAEndR for ages 22–24"],
        "missing_rows": [
            qwi.loc[qwi["agegrp"] == "A00", "EmpEnd"].isna().sum(),
            qwi.loc[qwi["agegrp"] == "A03", "HirAEndR"].isna().sum(),
        ],
    }
)

display(missing_summary)

ind_level: ['3']
ownercode: ['A05']
sex: ['0']
seasonadj: ['U']
agegrp: ['A00', 'A03']

Rows: 375068
States: 49
Industries: 96
Period: 2015Q1 to 2025Q4
Duplicate key rows: 0


,variable,missing_rows
0,EmpEnd for all ages,8150
1,HirAEndR for ages 22–24,24848


### AI Exposure Data Check

The AI-exposure dataset converts occupation-level exposure scores into national industry-level measures using ACS PUMS employment shares and person weights. Before merging it with QWI, I check that each industry appears once, that the exposure and coverage measures fall between zero and one, and that the industry codes match the QWI data. Industries without a defensible exposure estimate remain missing rather than receiving an assumed value.

In [8]:
# Check required columns
required_exposure_columns = {
    "industry",
    "industry_name",
    "ai_exposure",
    "ai_exposure_coverage",
    "eligible_person_weight",
    "matched_person_weight",
}

missing_columns = required_exposure_columns - set(exposure.columns)
assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"


# Check industry identifiers
duplicate_industries = exposure.duplicated("industry", keep=False).sum()

print("Industries:", exposure["industry"].nunique())
print("Duplicate industries:", duplicate_industries)

assert duplicate_industries == 0


# Check that the industry codes match QWI
qwi_industries = set(qwi["industry"].dropna())
exposure_industries = set(exposure["industry"].dropna())

print("QWI industries without an exposure row:", qwi_industries - exposure_industries)
print("Exposure industries not found in QWI:", exposure_industries - qwi_industries)

assert qwi_industries == exposure_industries


# Check the exposure values
assert exposure["ai_exposure"].dropna().between(0, 1).all()
assert exposure["ai_exposure_coverage"].dropna().between(0, 1).all()

print("Industries with an exposure score:", exposure["ai_exposure"].notna().sum())
print("Industries without an exposure score:", exposure["ai_exposure"].isna().sum())


# Review exposure coverage
display(
    exposure[["ai_exposure", "ai_exposure_coverage"]].describe()
)


# Show industries without a usable exposure score
missing_exposure = exposure.loc[
    exposure["ai_exposure"].isna(),
    ["industry", "industry_name"],
]

display(missing_exposure)

Industries: 96
Duplicate industries: 0
QWI industries without an exposure row: set()
Exposure industries not found in QWI: set()
Industries with an exposure score: 80
Industries without an exposure score: 16


,ai_exposure,ai_exposure_coverage
count,80.000000,80.000000
mean,0.381802,0.535200
std,0.103529,0.165674
min,0.126677,0.193776
25%,0.316714,0.423280
50%,0.377806,0.507937
75%,0.441073,0.687894
max,0.655173,0.883031


,industry,industry_name
9,236,Construction of Buildings
10,237,Heavy and Civil Engineering Construction
11,238,Specialty Trade Contractors
62,521,Monetary Authorities-Central Bank
64,523,"Securities, Commodity Contracts, and Other Fin..."
66,525,"Funds, Trusts, and Other Financial Vehicles"
69,533,Lessors of Nonfinancial Intangible Assets (exc...
71,551,Management of Companies and Enterprises
88,921,"Executive, Legislative, and Other General Gove..."
89,922,"Justice, Public Order, and Safety Activities"
